# Model 2 – 04 Ensemble disagreement analysis
**Why:** In the real-world app tests (20 Oct), the two ensemble members disagreed on HAND/LEG/OTHER for 5 images
(knee front, knee side, elbow, shoulder, skull) and DeiT-Small was right in all 5. Those 5 images were chosen by us,
so this notebook checks the pattern on held-out data that was NOT used for that observation.
**Question:** When MobileNetV3 and DeiT-Small disagree on HAND/LEG/OTHER, which one is right?
**Inputs:** outputs/model2/preds_mobilenetv3_{val,test}.csv, preds_deit_small_{val,test}.csv (saved by notebooks 02 and 03)
**Outputs:** printed tables only (no model is changed, no training).

In [1]:
# ===== SETTINGS =====
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/Research')
OUT_DIR    = DRIVE_ROOT / 'outputs' / 'model2'
SPLITS     = ['val', 'test']

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

Mounted at /content/drive


## Step 1 – Who is right when the two models disagree?

In [3]:
rows = []
for split in SPLITS:
    mb = pd.read_csv(OUT_DIR / f'preds_mobilenetv3_{split}.csv')
    dt = pd.read_csv(OUT_DIR / f'preds_deit_small_{split}.csv')
    assert (mb.filename == dt.filename).all(), 'row order differs!'
    dis = mb.pred_super != dt.pred_super                 # disagree on HAND/LEG/OTHER
    t = mb.true_super[dis]
    rows.append({'split': split, 'images': len(mb), 'disagreements': int(dis.sum()),
                 'MobileNetV3 right': int((mb.pred_super[dis] == t).sum()),
                 'DeiT-Small right': int((dt.pred_super[dis] == t).sum()),
                 'both wrong': int(((mb.pred_super[dis] != t) & (dt.pred_super[dis] != t)).sum())})
    print(f'\n=== {split}: disagreement cases ===')
    print(pd.DataFrame({'file': mb.filename[dis], 'true': t, 'MobileNetV3': mb.pred_super[dis],
                        'DeiT-Small': dt.pred_super[dis]}).to_string(index=False))
print('\n=== Summary ===')
print(pd.DataFrame(rows).to_string(index=False))


=== val: disagreement cases ===
                          file  true MobileNetV3 DeiT-Small
     fracture_forearm_0490.jpg  HAND        HAND        LEG
     fracture_forearm_0502.jpg  HAND        HAND        LEG
     fracture_forearm_0171.jpg  HAND        HAND        LEG
     fracture_forearm_0336.jpg  HAND        HAND        LEG
     hairline_forearm_0001.jpg  HAND        HAND        LEG
pathological_hip-knee_0013.jpg   LEG         LEG       HAND
  fracture_knee-ankle_0163.jpg   LEG         LEG       HAND
comminuted_knee-ankle_0004.jpg   LEG        HAND        LEG
  avulsion_knee-ankle_0011.jpg   LEG        HAND        LEG
   longitudinal_other_0001.jpg OTHER        HAND      OTHER
       fracture_other_0030.jpg OTHER       OTHER       HAND
       fracture_other_0016.jpg OTHER         LEG      OTHER
    dislocation_other_0002.jpg OTHER         LEG      OTHER

=== test: disagreement cases ===
                                                file  true MobileNetV3 DeiT-Small
           

In [4]:
## Step 2 – Effect of the routing rule (route only if both agree AND ensemble confidence >= 0.70)
import numpy as np
S = ['HAND', 'LEG', 'OTHER']
for split in SPLITS:
    mb = pd.read_csv(OUT_DIR / f'preds_mobilenetv3_{split}.csv')
    dt = pd.read_csv(OUT_DIR / f'preds_deit_small_{split}.csv')
    ens = (mb[[f'p_{s}' for s in S]].values + dt[[f'p_{s}' for s in S]].values) / 2
    pred = np.array(S)[ens.argmax(1)]
    correct = pred == mb.true_super.values
    routed = (mb.pred_super == dt.pred_super).values & (ens.max(1) >= 0.70)
    print(f'{split}: routed {routed.sum()}/{len(mb)} ({routed.mean():.1%}) | accuracy of routed {correct[routed].mean():.1%} '
          f'| wrongly routed {(routed & ~correct).sum()} | sent to review {(~routed).sum()} '
          f'| ensemble accuracy without the rule {correct.mean():.1%}')

val: routed 510/525 (97.1%) | accuracy of routed 98.2% | wrongly routed 9 | sent to review 15 | ensemble accuracy without the rule 96.8%
test: routed 508/519 (97.9%) | accuracy of routed 99.2% | wrongly routed 4 | sent to review 11 | ensemble accuracy without the rule 98.7%


## Summary – ensemble disagreement analysis
Disagreements on HAND/LEG/OTHER: val 13/525, test 9/519 (2.1% overall); in every case one model was right (both wrong: 0).
Who is right overall: MobileNetV3 11, DeiT-Small 11 → no single better model (the 5/5 real-world observation did not generalise).
By true class: HAND – MobileNetV3 7/8; LEG – DeiT 6/9; OTHER – DeiT 4/5 → complementary errors (CNN vs transformer),
which supports using an ensemble. Real-world disagreements (knees = LEG, skull = OTHER) match DeiT's strengths.
Routing rule (route only if both agree AND ensemble confidence ≥ 0.70, threshold fixed in advance):
| Split | Routed | Accuracy of routed | Wrongly routed | Sent to review | Wrongly routed without rule |
|---|---|---|---|---|---|
| val | 510/525 (97.1%) | 98.2% | 9 | 15 | 17 (96.8% acc) |
| test | 508/519 (97.9%) | 99.2% | 4 | 11 | 7 (98.7% acc) |
→ the rule roughly halves wrong routings at a cost of 2–3% human review.
Remaining errors = both models agree and are confident but wrong → needs more/better LEG data (future work).
Future work: class-aware weighting (trust DeiT more for LEG/OTHER), validated on a new independent set.